## Extractive Question Answering Using BERT

This notebook implements an extractive question answering pipeline based on the pre-trained **BERT-large** model. Given a question and a context document, the model identifies the most relevant answer span directly from the source text.

In [1]:
!pip install "numpy<2" pandas
!pip install torch==2.2.2 torchvision==0.17.2 torchaudio==2.2.2 --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.40.0

Looking in indexes: https://download.pytorch.org/whl/cu121


### Library Imports

In [2]:
import json
import re
import string
from pathlib import Path
import time

import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

### Configuration and Hyperparameters

The following cell defines the model identifier, file paths for input/output data, and key hyperparameters governing the tokenisation and answer extraction process. `MAX_LENGTH` and `DOC_STRIDE` control the sliding window mechanism used to handle contexts that exceed the model's maximum input length. `TOP_K_LOGITS` determines the number of candidate start and end positions considered during answer span selection.

In [3]:
MODEL_NAME = "bert-large-uncased-whole-word-masking-finetuned-squad"

INPUT_PATH = "input.json"
OUTPUT_PATH = "output.json"
DEV_PATH = "dev.json"
WRONG_OUTPUT_PATH = "wrong_predictions.json"

MAX_LENGTH = 384
DOC_STRIDE = 192
MAX_ANSWER_LENGTH = 12
TOP_K_LOGITS = 20

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

print("Model loaded:", MODEL_NAME)

c:\Users\MSI\miniconda3\envs\torchgpu\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Model loaded: bert-large-uncased-whole-word-masking-finetuned-squad


### Data Loading

In [5]:
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    input_data = json.load(f)

print("Loaded input records:", len(input_data))
print("First sample keys:", input_data[0].keys())

Loaded input records: 150
First sample keys: dict_keys(['question_id', 'question', 'document'])


### Output Normalisation

This function performs post-processing on extracted answer strings by removing extraneous whitespace and newline characters to ensure clean, consistent output formatting.

In [6]:
def clean_text(text):
    if text is None:
        return ""
    return str(text).strip()


def safe_output_normalize(text):
    if text is None:
        return ""

    text = str(text)
    text = text.replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()

    return text

### Answer Extraction with Sliding Window

The core inference function tokenises the question–context pair using a sliding window approach to accommodate long documents. For each resulting feature window, the top-*k* start and end logits are evaluated combinatorially. Candidate spans are filtered by length and positional validity, and the highest-scoring valid span is selected as the predicted answer.

In [7]:
def get_best_answer_from_features(question, context):
    question = clean_text(question)
    context = clean_text(context)

    if not question or not context:
        return ""

    inputs = tokenizer(
        question,
        context,
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
        return_tensors="pt"
    )

    input_ids = inputs["input_ids"].to(DEVICE)
    attention_mask = inputs["attention_mask"].to(DEVICE)
    offset_mapping = inputs["offset_mapping"]

    model_inputs = {
        "input_ids": input_ids,
        "attention_mask": attention_mask
    }

    if "token_type_ids" in inputs:
        model_inputs["token_type_ids"] = inputs["token_type_ids"].to(DEVICE)

    with torch.no_grad():
        outputs = model(**model_inputs)

    start_logits = outputs.start_logits.cpu()
    end_logits = outputs.end_logits.cpu()

    best_answer = ""
    best_score = float("-inf")

    for feature_idx in range(len(inputs["input_ids"])):
        offsets = offset_mapping[feature_idx]
        sequence_ids = inputs.sequence_ids(feature_idx)

        context_token_indexes = [
            idx for idx, seq_id in enumerate(sequence_ids)
            if seq_id == 1
        ]

        if not context_token_indexes:
            continue

        start_logit = start_logits[feature_idx]
        end_logit = end_logits[feature_idx]

        top_start = torch.topk(
            start_logit, 
            k=min(TOP_K_LOGITS, len(start_logit))
        ).indices.tolist()

        top_end = torch.topk(
            end_logit, 
            k=min(TOP_K_LOGITS, len(end_logit))
        ).indices.tolist()

        for start_index in top_start:
            for end_index in top_end:
                if start_index not in context_token_indexes:
                    continue
                if end_index not in context_token_indexes:
                    continue
                if end_index < start_index:
                    continue

                span_len = end_index - start_index + 1
                if span_len > MAX_ANSWER_LENGTH:
                    continue

                start_offset = offsets[start_index]
                end_offset = offsets[end_index]

                if start_offset is None or end_offset is None:
                    continue

                start_char, _ = start_offset.tolist()
                _, end_char = end_offset.tolist()

                if start_char >= end_char:
                    continue

                answer_text = context[start_char:end_char].strip()
                if not answer_text:
                    continue

                score = start_logit[start_index].item() + end_logit[end_index].item()

                if score > best_score:
                    best_score = score
                    best_answer = answer_text

    return best_answer.strip()

### Single-Example Prediction Wrapper

In [8]:
def predict_one(example):
    question = example["question"]
    context = example["document"]

    predicted_answer = get_best_answer_from_features(question, context)
    predicted_answer = safe_output_normalize(predicted_answer)

    return {
        "question_id": example["question_id"],
        "answer": predicted_answer
    }

### Batch Inference

The prediction function is applied iteratively over the entire input dataset to generate answer predictions for all question–context pairs.

In [9]:

predictions = []
total_start = time.time()

for i, example in enumerate(input_data, start=1):
    q_start = time.time()
    pred = predict_one(example)
    q_elapsed = time.time() - q_start
    predictions.append(pred)
    cumulative = time.time() - total_start
    print(f"[{i:>3}/{len(input_data)}] {example['question_id']}: "
          f"{q_elapsed:.2f}s | total: {cumulative:.1f}s | "
          f"ans: '{pred['answer'][:60]}'")

total_elapsed = time.time() - total_start
print(f"\nPredictions created: {len(predictions)}")
print(f"Total time: {total_elapsed:.1f}s | Avg: {total_elapsed/len(predictions):.2f}s per question")


[  1/150] q1: 0.72s | total: 0.7s | ans: 'improve infrastructure'
[  2/150] q2: 0.04s | total: 0.8s | ans: 'cave crickets'
[  3/150] q3: 0.04s | total: 0.8s | ans: 'Bhopal'
[  4/150] q4: 0.04s | total: 0.8s | ans: 'Baptists'
[  5/150] q5: 0.04s | total: 0.9s | ans: 'cooperation with the US, Russian, EU and Afghan authorities'
[  6/150] q6: 0.04s | total: 0.9s | ans: 'Mahasanghika'
[  7/150] q7: 0.04s | total: 1.0s | ans: 'Christian'
[  8/150] q8: 0.04s | total: 1.0s | ans: 'Orthodox Jewish'
[  9/150] q9: 0.04s | total: 1.0s | ans: 'harp'
[ 10/150] q10: 0.04s | total: 1.1s | ans: 'Maripol'
[ 11/150] q11: 0.04s | total: 1.1s | ans: 'E = mc²'
[ 12/150] q12: 0.04s | total: 1.2s | ans: 'eight'
[ 13/150] q13: 0.04s | total: 1.2s | ans: '905 AD'
[ 14/150] q14: 0.04s | total: 1.2s | ans: 'paragraph 6'
[ 15/150] q15: 0.04s | total: 1.3s | ans: 'limits'
[ 16/150] q16: 0.04s | total: 1.3s | ans: 'five'
[ 17/150] q17: 0.04s | total: 1.4s | ans: '1915'
[ 18/150] q18: 0.04s | total: 1.4s | ans: '21s

### Export Predictions

In [10]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, indent=2, ensure_ascii=False)

print(f"Saved predictions to {OUTPUT_PATH}")

Saved predictions to output.json
